# IIR Low-Pass Filter on IQ Signals with GNU Radio

This notebook implements an **IIR (Infinite Impulse Response) low-pass filter** on synthetic IQ signals using **GNU Radio** flowgraph blocks.

---

### Objectives
1. **Synthetic IQ Data**: Create a reproducible 2-component IQ dataset using the canonical shape `(N, 2, L)`.
2. **GNU Radio Type Consistency**: Address stream itemsize differences between complex streams (`gr_complex`, 8 bytes) and float blocks (`single_pole_iir_filter_ff`, 4 bytes).
3. **Flowgraph Execution**: Construct and run a `gr.top_block` pipeline using `vector_source_c`, `single_pole_iir_filter_ff`, and `vector_sink_c`.
4. **Verification & Testing**: Verify that output dimensions match input dimensions, measure power attenuation, and visualize the filtered output.

## 1. Architectural Note: GNU Radio Block Types & Stream Sizing

GNU Radio block suffixes designate stream types:
- `_c`: Complex single-precision float (`gr_complex`, 64 bits = 32-bit I + 32-bit Q; 8-byte itemsize).
- `_f` or `_ff`: Single-precision float (`float32`; 4-byte itemsize).

> [!IMPORTANT]
> Connecting `blocks.vector_source_c` directly to `blocks.single_pole_iir_filter_ff` causes an **itemsize mismatch** (8 bytes vs 4 bytes).
>
> **Type-Correct Topology**:
> 1. Demultiplex complex samples into independent I and Q float streams using `blocks.complex_to_float()`.
> 2. Filter both I and Q branches in parallel using two `blocks.single_pole_iir_filter_ff(alpha, 1)` blocks.
> 3. Multiplex filtered I and Q streams back into complex samples via `blocks.float_to_complex()`.
> 4. Sink to `blocks.vector_sink_c()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Verify GNU Radio availability
try:
    from gnuradio import gr, blocks
    HAS_GNURADIO = True
    print("GNU Radio imported successfully.")
except ImportError:
    HAS_GNURADIO = False
    print("NOTICE: GNU Radio is not found in the current Python environment.")
    print("Refer to AGENTS.md for instructions on creating a virtual environment with:")
    print("    python3 -m venv --system-site-packages .venv")

## 2. Generate Synthetic IQ Signal

We construct a synthetic dataset with:
- $N = 5$ examples
- $L = 1000$ time samples
- Canonical layout `(N, 2, L)`: axis 0 = examples, axis 1 = I/Q components, axis 2 = time.
- Content: baseband sinusoidal tone ($f_0 = 5\text{ Hz}$) corrupted by additive high-frequency Gaussian noise.

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

N, L = 5, 1000
t = np.linspace(0, 1, L, endpoint=False, dtype=np.float32)

# Baseband sinusoidal tone (f0 = 5 Hz)
f0 = 5.0
tone_I = np.cos(2 * np.pi * f0 * t, dtype=np.float32)
tone_Q = np.sin(2 * np.pi * f0 * t, dtype=np.float32)

# Replicate across N examples and add high-frequency noise
I = np.repeat(tone_I[np.newaxis, :], N, axis=0) + 0.6 * rng.standard_normal((N, L)).astype(np.float32)
Q = np.repeat(tone_Q[np.newaxis, :], N, axis=0) + 0.6 * rng.standard_normal((N, L)).astype(np.float32)

# Stack into canonical (N, 2, L) tensor
X = np.stack([I, Q], axis=1)  # shape: (5, 2, 1000)

# Prepare flattened complex stream for GNU Radio vector source
iq_complex_batch = I + 1j * Q
iq_flat = iq_complex_batch.flatten().astype(np.complex64).tolist()

print(f"Canonical shape X: {X.shape}, dtype: {X.dtype}")
print(f"Total complex samples for flowgraph: {len(iq_flat)}")

## 3. Flowgraph Design: `IIRFilterFlowgraph`

The single-pole IIR filter implements the discrete-time recurrence:
$$
y[n] = (1 - \alpha) \cdot y[n-1] + \alpha \cdot x[n]
$$
where $\alpha \in (0, 1]$ represents the pole factor (smoothing coefficient).

Below is the complete GNU Radio flowgraph topology using `blocks.vector_source_c`, `blocks.complex_to_float`, `blocks.single_pole_iir_filter_ff`, `blocks.float_to_complex`, and `blocks.vector_sink_c`.

In [ ]:
if HAS_GNURADIO:
    class IIRFilterFlowgraph(gr.top_block):
        def __init__(self, samples, alpha=0.1):
            super().__init__("IIR_IQ_Filter_Flowgraph")
            
            # 1. Complex vector source
            self.source = blocks.vector_source_c(samples)
            
            # 2. Demux complex stream to float I and Q
            self.c2f = blocks.complex_to_float()
            
            # 3. Two single-pole IIR filters (one for I, one for Q)
            self.iir_i = blocks.single_pole_iir_filter_ff(alpha, 1)
            self.iir_q = blocks.single_pole_iir_filter_ff(alpha, 1)
            
            # 4. Mux float streams back to complex
            self.f2c = blocks.float_to_complex()
            
            # 5. Complex vector sink
            self.sink = blocks.vector_sink_c()
            
            # Connect flowgraph topology
            self.connect(self.source, self.c2f)
            self.connect((self.c2f, 0), self.iir_i, (self.f2c, 0))
            self.connect((self.c2f, 1), self.iir_q, (self.f2c, 1))
            self.connect(self.f2c, self.sink)
else:
    # Architectural fallback: Identical discrete-time recurrence in pure NumPy
    class IIRFilterFlowgraph:
        def __init__(self, samples, alpha=0.1):
            self.samples = np.array(samples, dtype=np.complex64)
            self.alpha = alpha
            self.output = None
            
        def run(self):
            # Apply recursive single-pole IIR filter on I and Q independently
            filtered = np.zeros_like(self.samples)
            prev = 0.0 + 0.0j
            for idx, x_val in enumerate(self.samples):
                y_val = (1.0 - self.alpha) * prev + self.alpha * x_val
                filtered[idx] = y_val
                prev = y_val
            self.output = filtered
            
        class _Sink:
            def __init__(self, parent):
                self.parent = parent
            def data(self):
                return self.parent.output
                
        @property
        def sink(self):
            return self._Sink(self)
            
print("IIRFilterFlowgraph defined successfully.")

## 4. Run Flowgraph & Reshape Output

We execute the flowgraph, pull the filtered samples from the sink, and reconstruct the canonical `(N, 2, L)` tensor.

In [ ]:
# Instantiate and run flowgraph
fg = IIRFilterFlowgraph(iq_flat, alpha=0.1)
fg.run()

# Extract data from sink
raw_out = np.array(fg.sink.data(), dtype=np.complex64)

# Reshape back to (N, L) complex, then (N, 2, L) canonical float tensor
filtered_complex = raw_out.reshape(N, L)
X_filtered = np.stack([
    filtered_complex.real.astype(np.float32),
    filtered_complex.imag.astype(np.float32)
], axis=1)

print(f"Input shape:  {X.shape}")
print(f"Output shape: {X_filtered.shape}")
assert X.shape == X_filtered.shape, f"Mismatch: {X.shape} vs {X_filtered.shape}"
print("Shape verification PASSED: Output matches (N, 2, L) exactly.")

## 5. Statistical Verification & Power Analysis

We compute the average signal power across all examples:
$$
P = \frac{1}{N \cdot L} \sum_{n=0}^{N-1} \sum_{l=0}^{L-1} \left( I[n, l]^2 + Q[n, l]^2 \right)
$$

Since the input contains wideband Gaussian noise and the filter is low-pass, the output power must decrease ($P_{\text{filtered}} < P_{\text{original}}$).

In [ ]:
p_before = np.mean(X[:, 0, :]**2 + X[:, 1, :]**2)
p_after = np.mean(X_filtered[:, 0, :]**2 + X_filtered[:, 1, :]**2)
ratio = p_after / p_before

print(f"Signal Power BEFORE filtering: {p_before:.4f}")
print(f"Signal Power AFTER filtering:  {p_after:.4f}")
print(f"Power ratio (P_after / P_before): {ratio:.4f}")

# Assert filter effectiveness
assert p_after < p_before, "Test failed: Output power was not attenuated."
print("Filter test PASSED: High-frequency noise attenuated successfully.")

## 6. Time Trace & Constellation Visualizations

We plot the zoomed time traces for both components (I and Q) and compare the I/Q constellation diagram before and after filtering.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# In-phase (I) time trace
axes[0, 0].plot(t[:150], X[0, 0, :150], label="Raw I (Noisy)", color="lightsteelblue", alpha=0.8)
axes[0, 0].plot(t[:150], X_filtered[0, 0, :150], label="Filtered I", color="navy", linewidth=2)
axes[0, 0].set_title("In-phase (I) Component (First 150 samples)")
axes[0, 0].set_xlabel("Time [s]")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle="--", alpha=0.6)

# Quadrature (Q) time trace
axes[0, 1].plot(t[:150], X[0, 1, :150], label="Raw Q (Noisy)", color="moccasin", alpha=0.8)
axes[0, 1].plot(t[:150], X_filtered[0, 1, :150], label="Filtered Q", color="darkorange", linewidth=2)
axes[0, 1].set_title("Quadrature (Q) Component (First 150 samples)")
axes[0, 1].set_xlabel("Time [s]")
axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle="--", alpha=0.6)

# Constellation: Before filtering
axes[1, 0].scatter(X[0, 0, :], X[0, 1, :], s=10, alpha=0.4, color="crimson")
axes[1, 0].set_title("Constellation BEFORE Filtering (Noisy Circle)")
axes[1, 0].set_xlabel("In-phase (I)")
axes[1, 0].set_ylabel("Quadrature (Q)")
axes[1, 0].grid(True, linestyle="--", alpha=0.6)
axes[1, 0].axis("equal")

# Constellation: After filtering
axes[1, 1].scatter(X_filtered[0, 0, :], X_filtered[0, 1, :], s=10, alpha=0.4, color="teal")
axes[1, 1].set_title("Constellation AFTER Filtering (Clean Orbit)")
axes[1, 1].set_xlabel("In-phase (I)")
axes[1, 1].set_ylabel("Quadrature (Q)")
axes[1, 1].grid(True, linestyle="--", alpha=0.6)
axes[1, 1].axis("equal")

plt.tight_layout()
plt.show()